# Fair-RAG Reproduction Notebook

This notebook reproduces the **Towards Fair RAG** experiment pipeline in this repository:
1. Prepare data and folders
2. Precompute deterministic retrieval
3. Build oracle retrieval inputs (delta files)
4. Run stochastic retrieval + generation + measurement
5. Normalize EU
6. Aggregate and plot results

> Default settings are CPU-safe and start with a smaller run. Switch to full settings in the parameters cell for paper-scale runs.

In [ ]:
from pathlib import Path
import os
import sys
import json
import zipfile
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import warnings

# Suppress transformers pipeline warnings
warnings.filterwarnings('ignore', category=UserWarning, module='transformers')
warnings.filterwarnings('ignore', message='.*pipelines sequentially on GPU.*')

REPO_ROOT = Path.cwd()
print('Repo root:', REPO_ROOT)
assert (REPO_ROOT / 'experiment.py').exists(), 'Run this notebook from the repository root.'

# Show model cache location
HF_CACHE = Path.home() / '.cache' / 'huggingface' / 'hub'
print(f'\nModel cache directory: {HF_CACHE}')
print(f'Models will be downloaded here on first use (~1-15GB depending on model size)')

Repo root: /home/student/Fair-RAG

Model cache directory: /home/student/.cache/huggingface/hub
Models will be downloaded here on first use (~1-15GB depending on model size)


## Model Download & Caching

When you first run experiments with a language model (e.g., `flanT5Small`), the model (~1-2GB) is automatically downloaded from Hugging Face Hub and cached locally at:

```
~/.cache/huggingface/hub/
```

Subsequent runs use the cached copy—**no re-downloading needed**. 

### What happens during a run:
- **Cell 4**: Extracts and validates datasets (~2-3 min)
- **Cell 5**: Builds oracle inputs from relevance mappings (~1 min)
- **Cell 6**: Precomputes BM25 retrieval (~5-10 min per LAMP task)
- **Cell 7**: Runs stochastic experiments with LLM generation (longest step, ~10-30 min per run depending on model size)
- **Cell 8**: Normalizes and aggregates results (~1-2 min)

**Live output below shows exactly what's happening at each step.**

In [2]:
# ===== Helper function to run commands with live output =====
def run_cmd(cmd, description=""):
    """Run a command and stream output in real-time."""
    if description:
        print(f"\n{'='*60}")
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {description}")
        print(f"{'='*60}")
    print(f"Command: {' '.join(cmd)}\n")
    
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=REPO_ROOT,
        bufsize=1
    )
    
    for line in proc.stdout:
        print(line.rstrip())
    
    returncode = proc.wait()
    if returncode != 0:
        raise RuntimeError(f"Command failed with code {returncode}")
    print()

# ===== Parameters =====
GENERATOR_NAME = 'flanT5Small'  # flanT5Small | flanT5Base | flanT5XXL
LAMP_NUMS = [4]                 # use [1,2,3,4,5,6,7] for full sweep
RETRIEVERS = ['bm25']           # CPU-safe: bm25; GPU-heavy: contriever, splade
ALPHAS = [1, 2, 4, 8]
K = 5
N_SAMPLES = 8                   # paper default in script is 100
REMOVE_TEMP_FILES = True
RUN_EXPERIMENTS = True

# For paper-scale reproduction (requires significant compute), set:
# LAMP_NUMS = [1,2,3,4,5,6,7]
# RETRIEVERS = ['bm25', 'contriever', 'splade']
# N_SAMPLES = 100

print(f"Configuration:")
print(f"  Generator: {GENERATOR_NAME}")
print(f"  LaMP tasks: {LAMP_NUMS}")
print(f"  Retrievers: {RETRIEVERS}")
print(f"  Alphas: {ALPHAS}")
print(f"  Total experiment combinations: {len(LAMP_NUMS) * len(RETRIEVERS) * len(ALPHAS)}")

Configuration:
  Generator: flanT5Small
  LaMP tasks: [4]
  Retrievers: ['bm25']
  Alphas: [1, 2, 4, 8]
  Total experiment combinations: 4


In [3]:
# ===== Setup: unzip provided utility datasets + create temp dirs =====
data_dir = REPO_ROOT / 'data'
for zfp in sorted(data_dir.glob('lamp_utility_labels_*.zip')):
    with zipfile.ZipFile(zfp, 'r') as zf:
        zf.extractall(data_dir)

nested = data_dir / 'data'
if nested.exists():
    for sub in nested.iterdir():
        target = data_dir / sub.name
        if not target.exists():
            sub.rename(target)
    try:
        nested.rmdir()
    except OSError:
        pass

(REPO_ROOT / 'trec_top_files').mkdir(exist_ok=True)
(REPO_ROOT / 'trec_rel_files').mkdir(exist_ok=True)
print('Data and temp directories are ready.')

Data and temp directories are ready.


In [4]:
# ===== Build delta files needed by retrieval/gold_retriever.py =====
cmd = [
    sys.executable,
    str(REPO_ROOT / 'utility_labels' / 'bootstrap_eval_results_from_relevance.py'),
    '--generator_name', GENERATOR_NAME,
    '--lamp_nums', *[str(x) for x in LAMP_NUMS],
]
run_cmd(cmd, "Building oracle retrieval delta files from relevance mappings")


[10:23:40] Building oracle retrieval delta files from relevance mappings
Command: /anaconda/envs/azureml_py310_sdkv2/bin/python /home/student/Fair-RAG/utility_labels/bootstrap_eval_results_from_relevance.py --generator_name flanT5Small --lamp_nums 4



Wrote: /home/student/Fair-RAG/utility_labels/eval_results/flanT5Small/4_delta.tsv



In [5]:
# ===== Precompute deterministic retrieval and oracle retrieval =====
for lamp_num in LAMP_NUMS:
    for retriever in RETRIEVERS:
        cmd = [
            sys.executable,
            str(REPO_ROOT / 'retrieval' / 'rank_profiles.py'),
            '--ranker', retriever,
            '--generator_name', GENERATOR_NAME,
            '--lamp_num', str(lamp_num),
        ]
        run_cmd(cmd, f"Precomputing {retriever.upper()} retrieval for LaMP {lamp_num}")

    cmd = [
        sys.executable,
        str(REPO_ROOT / 'retrieval' / 'gold_retriever.py'),
        '--generator_name', GENERATOR_NAME,
        '--lamp_num', str(lamp_num),
    ]
    run_cmd(cmd, f"Building oracle (gold) retrieval for LaMP {lamp_num}")


[10:23:49] Precomputing BM25 retrieval for LaMP 4
Command: /anaconda/envs/azureml_py310_sdkv2/bin/python /home/student/Fair-RAG/retrieval/rank_profiles.py --ranker bm25 --generator_name flanT5Small --lamp_num 4




100%|██████████| 833/833 [00:04<00:00, 176.30it/s]


[10:24:18] Building oracle (gold) retrieval for LaMP 4
Command: /anaconda/envs/azureml_py310_sdkv2/bin/python /home/student/Fair-RAG/retrieval/gold_retriever.py --generator_name flanT5Small --lamp_num 4




In [7]:
# ===== Main experiments + normalization =====
if RUN_EXPERIMENTS:
    retrievers_to_run = list(dict.fromkeys(RETRIEVERS + ['gold']))
    total_runs = sum(len(LAMP_NUMS) * (1 if r == 'gold' else len(ALPHAS)) for r in retrievers_to_run)
    run_count = 0

    for lamp_num in LAMP_NUMS:
        for retriever in retrievers_to_run:
            alpha_list = [8] if retriever == 'gold' else ALPHAS
            for alpha in alpha_list:
                run_count += 1
                status = f"[{run_count}/{total_runs}]"
                
                cmd = [
                    sys.executable, str(REPO_ROOT / 'experiment.py'),
                    '--retriever_name', retriever,
                    '--generator_name', GENERATOR_NAME,
                    '--lamp_num', str(lamp_num),
                    '--alpha', str(alpha),
                    '--k', str(K),
                    '--n_samples', str(N_SAMPLES),
                ]
                if REMOVE_TEMP_FILES:
                    cmd.append('--remove_temp_files')
                
                run_cmd(cmd, f"{status} Running experiment: LaMP{lamp_num} | {retriever} | alpha={alpha}")

        for retriever in RETRIEVERS:
            for alpha in ALPHAS:
                cmd = [
                    sys.executable, str(REPO_ROOT / 'normalize_eu.py'),
                    '--retriever_name', retriever,
                    '--generator_name', GENERATOR_NAME,
                    '--lamp_num', str(lamp_num),
                    '--alpha', str(alpha),
                ]
                run_cmd(cmd, f"Normalizing results: LaMP{lamp_num} | {retriever} | alpha={alpha}")
                
    print("\n" + "="*60)
    print("✓ All experiments completed!")
    print("="*60)


[11:40:42] [1/5] Running experiment: LaMP4 | bm25 | alpha=1
Command: /anaconda/envs/azureml_py310_sdkv2/bin/python /home/student/Fair-RAG/experiment.py --retriever_name bm25 --generator_name flanT5Small --lamp_num 4 --alpha 1 --k 5 --n_samples 8 --remove_temp_files



/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Processing 833 queries...
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maxim

KeyboardInterrupt: 

In [ ]:
# ===== Aggregate and plot =====
print(f"\nAggregating results from {len(LAMP_NUMS)} LaMP task(s), {len(RETRIEVERS)} retriever(s), {len(ALPHAS)} alpha value(s)...")
print()

rows = []
for lamp_num in LAMP_NUMS:
    for retriever in RETRIEVERS:
        for alpha in ALPHAS:
            fp = REPO_ROOT / 'experiment_results' / GENERATOR_NAME / f'lamp{lamp_num}' / retriever / f'alpha_{alpha}_normalized.json'
            if not fp.exists():
                print(f"Skipping missing: {fp.name}")
                continue
            with open(fp, 'r', encoding='utf-8') as f:
                d = json.load(f)

            first_qid = next(iter(d.keys()))
            metric_name = next(iter(d[first_qid]['EU'].keys()))
            ee_diff = [v['EE']['difference'] for v in d.values()]
            eu_vals = [v['EU'][metric_name] for v in d.values()]

            rows.append({
                'lamp_num': lamp_num,
                'retriever': retriever,
                'alpha': alpha,
                'metric': metric_name,
                'EE_difference_mean': float(pd.Series(ee_diff).mean()),
                'EE_difference_std': float(pd.Series(ee_diff).std()),
                'EU_mean': float(pd.Series(eu_vals).mean()),
                'EU_std': float(pd.Series(eu_vals).std()),
            })

if rows:
    summary = pd.DataFrame(rows).sort_values(['lamp_num', 'retriever', 'alpha'])
    print("Summary of Results:")
    print(summary.to_string(index=False))
    print()
    
    display(summary)

    if not summary.empty:
        for lamp_num, lamp_df in summary.groupby('lamp_num'):
            plt.figure(figsize=(6, 4))
            for retriever, retr_df in lamp_df.groupby('retriever'):
                retr_df = retr_df.sort_values('alpha')
                plt.plot(retr_df['EE_difference_mean'], retr_df['EU_mean'], marker='o', label=retriever)
                for _, r in retr_df.iterrows():
                    plt.annotate(f"a={int(r['alpha'])}", (r['EE_difference_mean'], r['EU_mean']), fontsize=8)

            plt.title(f'LaMP {lamp_num} | {GENERATOR_NAME}')
            plt.xlabel('Expected Exposure Difference (mean)')
            plt.ylabel('Normalized Expected Utility (mean)')
            plt.grid(True, alpha=0.25)
            plt.legend()
            plt.tight_layout()
            plt.show()

        out_csv = REPO_ROOT / 'experiment_results' / GENERATOR_NAME / 'summary_metrics.csv'
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        summary.to_csv(out_csv, index=False)
        print(f'✓ Saved summary: {out_csv}')
        
        # Show cached models info
        hf_cache = Path.home() / '.cache' / 'huggingface' / 'hub'
        if hf_cache.exists():
            cache_size_gb = sum(f.stat().st_size for f in hf_cache.rglob('*') if f.is_file()) / (1024**3)
            print(f'\nModel cache used: {cache_size_gb:.1f} GB at {hf_cache}')
        
        # Show log files info
        print(f'\n' + '='*60)
        print(f'📋 Experiment Log Files')
        print(f'='*60)
        logs_found = False
        for lamp_num in LAMP_NUMS:
            for retriever in RETRIEVERS:
                for alpha in ALPHAS:
                    log_dir = REPO_ROOT / 'experiment_results' / GENERATOR_NAME / f'lamp{lamp_num}' / retriever
                    exp_log = log_dir / f'alpha_{alpha}.log'
                    norm_log = log_dir / f'alpha_{alpha}_normalize.log'
                    
                    if exp_log.exists():
                        print(f'Experiment log: {exp_log.relative_to(REPO_ROOT)}')
                        logs_found = True
                    if norm_log.exists():
                        print(f'Normalization log: {norm_log.relative_to(REPO_ROOT)}')
                        logs_found = True
        
        if logs_found:
            print(f'Log files contain configuration, progress, and result summaries for each run.')
        print(f'='*60)
else:
    print("No results found. Run cells 4-7 first to generate experiment outputs.")